# 04 · Comparative Summary

This notebook synthesises results from notebooks 02 and 03 into a single cross-metric
comparison suitable for the thesis results chapter.

**Units:**
- Energy → **J** (Joules)
- Time → **ms** (milliseconds)
- EDP → **J·ms** (Joule-milliseconds)

**Outputs:**
- `outputs/ranking_summary.csv` — full multi-metric ranking table
- Normalised metrics heatmap
- Top 3 / Bottom 3 per metric
- Key findings bullet list

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))   # make the shared style module importable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from itertools import combinations

import plot_style as ps
ps.apply_style()

# Canonical constants — single source: plot_style.
COL_CPU_ENERGY, COL_MEM_ENERGY = ps.COL_CPU_ENERGY, ps.COL_MEM_ENERGY
COL_TIME                       = ps.COL_TIME
COL_CPU_CARBON, COL_MEM_CARBON = ps.COL_CPU_CARBON, ps.COL_MEM_CARBON
PARADIGM        = ps.PARADIGM
PARADIGM_COLORS = ps.PARADIGM_COLORS
PARADIGM_ORDER  = ps.PARADIGM_ORDER
MEANPROPS       = ps.MEANPROPS
ALPHA           = ps.ALPHA

OUTPUTS_DIR = Path('outputs'); OUTPUTS_DIR.mkdir(exist_ok=True)

# Single source of truth: per-run rows (df) + per-cell means with EDP (df_mean).
df      = ps.load_runs()
df_mean = ps.cell_means(df)

def lang_means(cols):
    """Per-language two-step mean (equal benchmark weight) for column(s) `cols`."""
    return ps.lang_means(df_mean, cols)

print(f"Runs: {df.shape} | Cell-means: {df_mean.shape} | "
      f"{df['language'].nunique()} languages \u00d7 {df['benchmark'].nunique()} benchmarks")
df_mean.head(3)

## 1. Multi-Metric Ranking Table

Each language is ranked (1 = best) on four metrics:
- Mean CPU Energy (J)
- Mean Memory Energy (J)
- Mean Execution Time (ms)
- Mean EDP — (CPU + Memory Energy) × Time (J·ms)

Values use benchmark-level means averaged across all 8 benchmarks (equal benchmark weight).
The overall rank is the average of the four individual ranks.

In [ ]:
# Two-step mean (equal benchmark weight): df_mean already holds the per-cell mean
# energy/time and a per-cell EDP column, so we only average the 8 cells per language.
agg = df_mean.groupby('language').agg(
    paradigm     = ('paradigm', 'first'),
    cpu_mean_J   = (COL_CPU_ENERGY, 'mean'),
    mem_mean_J   = (COL_MEM_ENERGY, 'mean'),
    time_mean_ms = (COL_TIME, 'mean'),
    edp_mean_Jms = ('EDP', 'mean'),
).round(4)

agg['cpu_rank']  = agg['cpu_mean_J'].rank().astype(int)
agg['mem_rank']  = agg['mem_mean_J'].rank().astype(int)
agg['time_rank'] = agg['time_mean_ms'].rank().astype(int)
agg['edp_rank']  = agg['edp_mean_Jms'].rank().astype(int)
agg['overall_rank'] = (
    (agg['cpu_rank'] + agg['mem_rank'] + agg['time_rank'] + agg['edp_rank']) / 4
).round(2)

ranking = agg.sort_values('edp_rank')
ranking.index.name = 'Language'
ranking[['paradigm', 'cpu_mean_J', 'mem_mean_J', 'time_mean_ms', 'edp_mean_Jms',
         'cpu_rank', 'mem_rank', 'time_rank', 'edp_rank', 'overall_rank']]

In [ ]:
export = ranking[['paradigm', 'cpu_mean_J', 'mem_mean_J', 'time_mean_ms',
                  'edp_mean_Jms', 'cpu_rank', 'mem_rank', 'time_rank',
                  'edp_rank', 'overall_rank']].copy()
export.columns = ['Paradigm', 'CPU Energy (J)', 'Mem Energy (J)', 'Time (ms)',
                  'EDP (J·ms)', 'CPU Rank', 'Mem Rank', 'Time Rank', 'EDP Rank',
                  'Overall Rank']
export.to_csv(OUTPUTS_DIR / 'ranking_summary.csv')
print(f"Saved → {OUTPUTS_DIR / 'ranking_summary.csv'}")
export

## 2. Normalized Efficiency Comparison

Each metric is expressed as a **ratio relative to the best (lowest) language** — so 1.0 = best
and e.g. 5.0 means this language consumes 5× more CPU energy / RAM energy / time than the
most efficient language. Values are computed from the **mean** across all 8 benchmarks,
matching the CLBG presentation style.

In [ ]:
# Two-step mean (equal benchmark weight) straight from the per-cell means in df_mean.
norm_agg = lang_means([COL_CPU_ENERGY, COL_MEM_ENERGY, COL_TIME])

norm_agg['CPU Energy Normalized'] = (norm_agg[COL_CPU_ENERGY] / norm_agg[COL_CPU_ENERGY].min()).round(2)
norm_agg['RAM Energy Normalized'] = (norm_agg[COL_MEM_ENERGY] / norm_agg[COL_MEM_ENERGY].min()).round(2)
norm_agg['Time Normalized']       = (norm_agg[COL_TIME]        / norm_agg[COL_TIME].min()).round(2)

norm_ranking = (
    norm_agg[['CPU Energy Normalized', 'RAM Energy Normalized', 'Time Normalized']]
    .sort_values('CPU Energy Normalized')
    .reset_index()
    .rename(columns={'language': 'Language'})
)

norm_ranking.index = norm_ranking.index + 1  # rank starts at 1
norm_ranking.index.name = 'Rank'

norm_ranking

## 4. Normalised Metrics Heatmap

All four metrics normalised to [0, 1] per column. Lower values (greener) are better.
Enables direct visual comparison of language profiles across all metrics in human-readable units.

In [ ]:
metrics = {
    'CPU Energy (J)': COL_CPU_ENERGY,
    'Mem Energy (J)': COL_MEM_ENERGY,
    'Time (ms)':       COL_TIME,
    'EDP (J·ms)':      'EDP',
}
norm_df = pd.DataFrame(index=ranking.index)
for label, col in metrics.items():
    vals = df_mean.groupby('language')[col].mean()
    norm_df[label] = (vals - vals.min()) / (vals.max() - vals.min())

norm_df = norm_df.loc[ranking.index]  # order by overall rank

fig, ax = plt.subplots(figsize=(9, 10))
sns.heatmap(norm_df, annot=True, fmt='.2f',
            cmap=sns.diverging_palette(120, 10, as_cmap=True),
            center=0.5, vmin=0, vmax=1, ax=ax,
            linewidths=0.5, cbar_kws={'label': '0 = best, 1 = worst'})
ax.set_title('Normalised Metric Heatmap (sorted by Overall Rank)', fontsize=12)
ax.set_ylabel('Language (best → worst overall)')
plt.tight_layout()
ps.save_fig(fig, '04_normalised_heatmap')
plt.show()

> **Takeaway:** sorted by overall rank, the heatmap shows AOT languages green across every metric and interpreted languages red — the efficiency gap is consistent, not metric-specific.

## 5. Top 3 / Bottom 3 per Metric

Quick-reference tables for the most and least efficient languages on each metric.
All values in human-readable units (J, ms, J·ms).

In [ ]:
metric_cols = {
    'CPU Energy (J)':    COL_CPU_ENERGY,
    'Memory Energy (J)': COL_MEM_ENERGY,
    'Execution Time (ms)':COL_TIME,
    'EDP (J·ms)':         'EDP',
}
units = {'CPU Energy (J)': 'J', 'Memory Energy (J)': 'J',
         'Execution Time (ms)': 'ms', 'EDP (J·ms)': 'J·ms'}

for label, col in metric_cols.items():
    mean_series = df_mean.groupby('language')[col].mean().sort_values()
    top3 = mean_series.head(3)
    bot3 = mean_series.tail(3)
    unit = units[label]
    print(f"\n{'─'*55}")
    print(f"  {label}")
    print(f"  Top 3 (most efficient):")
    for lang, val in top3.items():
        print(f"    • {lang:12s} ({PARADIGM[lang]:12s})  {val:>10.3f} {unit}")
    print(f"  Bottom 3 (least efficient):")
    for lang, val in bot3.items():
        print(f"    • {lang:12s} ({PARADIGM[lang]:12s})  {val:>10.3f} {unit}")

## 6. Key Findings

A bullet-point summary of the main findings in human-readable units (J, ms, J·ms),
suitable for direct citation in the thesis.

In [ ]:
cpu_rank_ser  = df_mean.groupby('language')[COL_CPU_ENERGY].mean().sort_values()
time_rank_ser = df_mean.groupby('language')[COL_TIME].mean().sort_values()
edp_rank_ser  = df_mean.groupby('language')['EDP'].mean().sort_values()

aot_cpu  = df_mean[df_mean['paradigm']=='AOT'][COL_CPU_ENERGY].mean()
jit_cpu  = df_mean[df_mean['paradigm']=='JIT'][COL_CPU_ENERGY].mean()
int_cpu  = df_mean[df_mean['paradigm']=='Interpreted'][COL_CPU_ENERGY].mean()
aot_time = df_mean[df_mean['paradigm']=='AOT'][COL_TIME].mean()
jit_time = df_mean[df_mean['paradigm']=='JIT'][COL_TIME].mean()
int_time = df_mean[df_mean['paradigm']=='Interpreted'][COL_TIME].mean()

print("""
KEY FINDINGS — Benchmark Energy & Time Analysis (18 languages, 8 CLBG benchmarks)
═════════════════════════════════════════════════════════════════════════════════""")

print(f"""
CPU ENERGY (unit: J)
  • Most efficient:   {', '.join(cpu_rank_ser.head(3).index)}
  • Least efficient:  {', '.join(cpu_rank_ser.tail(3).index)}
  • AOT mean:         {aot_cpu:.2f} J
  • JIT mean:         {jit_cpu:.2f} J  ({jit_cpu/aot_cpu:.1f}× AOT)
  • Interpreted mean: {int_cpu:.2f} J  ({int_cpu/aot_cpu:.1f}× AOT)

EXECUTION TIME (unit: ms)
  • Fastest:          {', '.join(time_rank_ser.head(3).index)}
  • Slowest:          {', '.join(time_rank_ser.tail(3).index)}
  • AOT mean:         {aot_time:.2f} ms
  • JIT mean:         {jit_time:.2f} ms  ({jit_time/aot_time:.1f}× AOT)
  • Interpreted mean: {int_time:.2f} ms  ({int_time/aot_time:.1f}× AOT)

ENERGY-DELAY PRODUCT (unit: J·ms)
  • Best EDP:         {', '.join(edp_rank_ser.head(3).index)}
  • Worst EDP:        {', '.join(edp_rank_ser.tail(3).index)}
  • Best mean EDP:    {edp_rank_ser.iloc[0]:.2f} J·ms  ({edp_rank_ser.index[0]})
  • Worst mean EDP:   {edp_rank_ser.iloc[-1]:.2f} J·ms ({edp_rank_ser.index[-1]})

Note: All values are two-step means (per-cell mean → mean across 8 CLBG benchmarks,
equal benchmark weight). Use non-parametric tests (notebooks 02–03) for paradigm distributions.
""")